# 05. Witness（証拠）の生成

**前提ファイル**: `input.json`, `network.compiled`
（`01_onnx.ipynb`, `04_compile.ipynb` まで実行済み）

In [ ]:
try:
    import google.colab, subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl", "onnx", "torch", "torchvision"])
except ImportError:
    pass

In [ ]:
import ezkl, json, os

compiled_model_path = 'network.compiled'
data_path           = 'input.json'
witness_path        = 'witness.json'

for f in [compiled_model_path, data_path]:
    assert os.path.exists(f), f"{f} がありません。"

## Witness とは

RareSkills では「**witness ベクトル `a`** = すべての制約を満たす変数の値」と学びました。

EZKL の witness は、実際の入力データを回路に通したときの：
- 入力値
- 各層の中間計算値
- 最終出力値

をすべて記録したものです。証明者はこの値を使って「正しく計算した」という証明（proof）を作ります。

In [ ]:
res = ezkl.gen_witness(data_path, compiled_model_path, witness_path)
assert os.path.exists(witness_path)
print("Witness 生成完了")

## witness.json の中身を確認する

In [ ]:
with open(witness_path) as f:
    w = json.load(f)

print(f"キー: {list(w.keys())}\n")

if 'inputs' in w:
    for i, inp in enumerate(w['inputs']):
        print(f"inputs[{i}]  : {len(inp)} 個の値（入力画像のピクセル値）")

if 'outputs' in w:
    classes = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']
    for i, out in enumerate(w['outputs']):
        print(f"outputs[{i}] : {len(out)} 個の値（10クラスのスコア）")
        predicted = out.index(max(out))
        print(f"予測クラス  : [{predicted}] {classes[predicted]}")

## Witness が必要な理由

```
Witness（計算証拠）
    + ZK 回路（制約の集まり）
    + 証明鍵 PK
         ↓
     proof.json（ZK 証明）
```

検証者は **proof だけ**で確認できます。Witness の中身（中間計算値）は検証に不要です。これが「ゼロ知識」の本質です。

---
次: `06_setup.ipynb` で SRS のダウンロードと鍵生成（Trusted Setup）を行います。